# Fake News Prediction Rebuild — Step 4: Vectorize, Split, Train, Compare

Two evaluation upgrades over the old notebook:

1. **Stratified holdout test set carved out first, untouched here.** Reserved for Step 8's final results table, not used for model selection, so that number isn't the same number we tuned against.
2. **TF-IDF fit inside the cross-validation pipeline, not on the whole dataset beforehand.** Vectorizing everything first and splitting after leaks each fold's own vocabulary and IDF weights into what looks like "unseen" validation data. A `Pipeline` inside `cross_validate` refits the vectorizer on each fold's training slice only.

Comparison: title-only vs title+text (the equivalent of the old with/without-author check, since this dataset has no author column). If title+text doesn't clearly beat title-only, that's worth knowing before shipping the bigger feature set.

**Leakage history for this notebook, so a future read of this doesn't have to reconstruct it from chat:**
- Subject excluded entirely (Step 2 EDA: 100% of subjects map to exactly one label).
- Leading Reuters dateline stripped (`src/preprocess.py`, `strip_reuters_dateline`).
- First CV run on `title_text` hit 98.88% accuracy — too clean to trust. Coefficient inspection on `LinearSVC` showed `reuter` and `edt` still among the strongest real-news indicators, and 13.92% of articles turned out to be duplicate text, skewed almost entirely fake (10,954 fake duplicates vs 438 real). Both fixed below: `strip_boilerplate()` in `preprocess.py` now catches mid-article "Reuters" self-references (not just the parenthesized dateline), embedded URLs, tweet embeds, image-credit captions, and EDT/EST-style timestamps; duplicates are dropped before any splitting happens.

In [1]:
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from src.preprocess import build_content

RANDOM_STATE = 42

## Load, dedupe, build content

Load raw CSVs, label, concat, shuffle, then drop duplicate article bodies **before** anything order- or split-sensitive touches the data — duplicates that survive into both train and validation folds let the model partly recognize memorized text instead of generalizing, and that effect isn't symmetric here (duplicates are ~96% fake-labeled).

Cached to `../data/content_cache.pkl` since building `content_title_text` (full stemming pass over every article body) takes several minutes. **If `src/preprocess.py` changes again, delete `content_cache.pkl` before rerunning this cell** — the cache has no way to know the cleaning logic changed, it'll just happily serve stale content.

In [2]:
CACHE_PATH = '../data/content_cache.pkl'

if os.path.exists(CACHE_PATH):
    df = pd.read_pickle(CACHE_PATH)
    print("loaded from cache")
else:
    fake_df = pd.read_csv('../data/Fake.csv')
    true_df = pd.read_csv('../data/True.csv')
    fake_df['label'] = 1
    true_df['label'] = 0

    df = pd.concat([fake_df, true_df], ignore_index=True)
    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    # Dedupe AFTER shuffling, so which copy of a duplicate survives is
    # effectively arbitrary rather than biased toward whichever CSV
    # loaded first.
    before = len(df)
    df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
    after = len(df)
    print(f"Dropped {before - after} duplicate-text rows ({(before - after) / before * 100:.2f}%)")

    df['content_title_only'] = df.apply(
        lambda r: build_content(r['title'], r['text'], include_text=False), axis=1
    )
    df['content_title_text'] = df.apply(
        lambda r: build_content(r['title'], r['text'], include_text=True), axis=1
    )

    df.to_pickle(CACHE_PATH)
    print("computed and cached")

print(df.shape)
print(df['label'].value_counts(normalize=True).round(3))
df[['content_title_only', 'content_title_text']].head(2)

Dropped 6252 duplicate-text rows (13.92%)
computed and cached
(38646, 7)
label
0    0.548
1    0.452
Name: proportion, dtype: float64


,content_title_only,content_title_text
0,ben stein call th circuit court commit coup ta...,ben stein call th circuit court commit coup ta...
1,trump drop steve bannon nation secur council,trump drop steve bannon nation secur council u...


## Sanity check: cleaning artifacts

This is what caught the leak on the first pass — a 20-row spot check is ~10 seconds versus a ~30-minute full run, so this runs before anything expensive every time `preprocess.py` changes. Should come back empty. If it doesn't, fix `preprocess.py`, restart the kernel, delete `content_cache.pkl`, and rerun the load cell above before touching anything below this.

In [3]:
sample = df[df['label'] == 0].sample(20, random_state=RANDOM_STATE)

flagged = 0
for _, row in sample.iterrows():
    cleaned = build_content(row['title'], row['text'], include_text=True)
    if 'reuter' in cleaned or 'edt' in cleaned:
        flagged += 1
        print("STILL PRESENT:", row.name)
        print(cleaned[:200])
        print('---')

print(f"\n{flagged} / 20 flagged")
assert df['text'].duplicated().sum() == 0, "duplicates still present after dedupe"


0 / 20 flagged


## Holdout split

15% held out, stratified on label, untouched until Step 8. Everything below trains and compares only on the remaining 85%.

In [4]:
train_df, holdout_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df['label'],
    random_state=RANDOM_STATE,
)

train_df.to_pickle('../data/train_holdout_split_train.pkl')
holdout_df.to_pickle('../data/train_holdout_split_holdout.pkl')

print(f"train: {train_df.shape}, holdout: {holdout_df.shape}")
print(train_df['label'].value_counts(normalize=True).round(3))
print(holdout_df['label'].value_counts(normalize=True).round(3))

train: (32849, 7), holdout: (5797, 7)
label
0    0.548
1    0.452
Name: proportion, dtype: float64
label
0    0.548
1    0.452
Name: proportion, dtype: float64


## Cross-validated comparison

5-fold stratified CV, TF-IDF refit inside each fold via `Pipeline`. Content columns pulled out with `.tolist()`, not `.values` — pandas backs these text columns with PyArrow's `ChunkedArray`, which only supports scalar indexing, and sklearn's CV fold splitting does fancy (array) indexing, which `ChunkedArray` raises on. A plain Python list sidesteps it entirely.

If this is too slow locally (Random Forest on ~32k documents per fold, 5 folds, 2 variants, 3 models = 30 fits), drop to `n_splits=3` or `RandomForestClassifier(n_estimators=50)` — note it in the README if you do, don't silently change it.

In [5]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'LinearSVC': LinearSVC(random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
}

variants = {
    'title_only': 'content_title_only',
    'title_text': 'content_title_text',
}

scoring = ['accuracy', 'f1', 'precision', 'recall']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

y_train = train_df['label'].values

results = []

for variant_name, col in variants.items():
    X_train = train_df[col].tolist()
    for model_name, model in models.items():
        pipe = Pipeline([
            ('tfidf', TfidfVectorizer()),
            ('clf', model),
        ])
        cv_result = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring, n_jobs=1)
        results.append({
            'variant': variant_name,
            'model': model_name,
            'accuracy_mean': cv_result['test_accuracy'].mean(),
            'accuracy_std': cv_result['test_accuracy'].std(),
            'f1_mean': cv_result['test_f1'].mean(),
            'precision_mean': cv_result['test_precision'].mean(),
            'recall_mean': cv_result['test_recall'].mean(),
        })
        print(f"done: {variant_name} / {model_name}")

results_df = pd.DataFrame(results).round(4)
results_df.sort_values(['variant', 'f1_mean'], ascending=[True, False])

done: title_only / LogisticRegression
done: title_only / LinearSVC
done: title_only / RandomForest
done: title_text / LogisticRegression
done: title_text / LinearSVC
done: title_text / RandomForest


,variant,model,accuracy_mean,accuracy_std,f1_mean,precision_mean,recall_mean
1,title_only,LinearSVC,0.9385,0.0013,0.9312,0.9406,0.9221
0,title_only,LogisticRegression,0.9340,0.0036,0.9254,0.9454,0.9062
2,title_only,RandomForest,0.9266,0.0016,0.9166,0.9410,0.8935
4,title_text,LinearSVC,0.9831,0.0014,0.9813,0.9850,0.9776
3,title_text,LogisticRegression,0.9758,0.0007,0.9730,0.9810,0.9652
5,title_text,RandomForest,0.9647,0.0022,0.9605,0.9717,0.9495


## Model selection

**Chosen: `title_text` + `LinearSVC`.**

CV results (5-fold, post dedupe + boilerplate/dateline/self-reference
stripping): accuracy 0.9831, F1 0.9813, precision 0.9850, recall 0.9776.

- **title+text over title-only:** 98.31% vs 93.85% accuracy on the same
  model. Real gap, not noise.
- **LinearSVC over Logistic Regression and Random Forest:** wins on
  accuracy, F1, and precision in both variants, with a tight
  `accuracy_std` (0.0014) across folds.
- **Known residual limitation, documented rather than chased further:**
  `edt` still shows up as a weak real-news indicator (4th-weakest term
  on that side) after three rounds of leak fixes (Subject exclusion,
  Reuters dateline/self-reference stripping, duplicate removal,
  boilerplate stripping). It's minor compared to what's already been
  removed and will be noted as a known limitation in the README rather
  than a fourth cleaning pass.

## Diagnostics: inspect what the model learned

This is the cell that caught three rounds of leakage: duplicate
articles (13.92%, skewed fake), mid-article "Reuters" self-references,
and image-caption/URL/tweet-embed boilerplate (`via`, `getti`, `pic`,
`com`, `featur`) — none of which are genuine content signal. Rerun this
any time the CV numbers look suspiciously good, not just once.

Resolved: `breitbart` was flagged in earlier runs as a strong
fake-indicator term and deliberately left in rather than stripped —
enumerating outlet names by hand doesn't generalize and just hides a
real limitation of the corpus instead of documenting it. It no longer
appears in the top 20 after the dedupe/boilerplate fixes, so no README
caveat needed for it specifically.

Known residual, accepted rather than chased further: `edt` still shows
up as a weak real-news indicator (4th-weakest term on that side). Minor
compared to what's already been removed. Documented as a known
limitation in the README rather than a fourth cleaning pass.

In [6]:
X_train_full = train_df['content_title_text'].tolist()
y_train_full = train_df['label'].values

diag_pipe = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LinearSVC(random_state=RANDOM_STATE)),
])
diag_pipe.fit(X_train_full, y_train_full)

feature_names = diag_pipe.named_steps['tfidf'].get_feature_names_out()
coefs = diag_pipe.named_steps['clf'].coef_[0]

top_fake = np.argsort(coefs)[-20:][::-1]
top_real = np.argsort(coefs)[:20]

print("Top terms pushing toward FAKE (label=1):")
for i in top_fake:
    print(f"  {feature_names[i]:20s} {coefs[i]:.3f}")

print("\nTop terms pushing toward REAL (label=0):")
for i in top_real:
    print(f"  {feature_names[i]:20s} {coefs[i]:.3f}")

Top terms pushing toward FAKE (label=1):
  via                  7.996
  gop                  5.312
  read                 4.885
  video                4.764
  us                   3.501
  sen                  3.374
  rep                  3.203
  mr                   3.084
  watch                3.008
  break                2.994
  even                 2.887
  hillari              2.691
  wire                 2.680
  reportedli           2.628
  entir                2.466
  actual               2.213
  gov                  2.206
  daili                2.200
  reveal               2.143
  america              2.118

Top terms pushing toward REAL (label=0):
  said                 -8.452
  wednesday            -4.058
  tuesday              -3.570
  thursday             -3.346
  nov                  -3.175
  friday               -3.133
  monday               -2.914
  factbox              -2.735
  rival                -2.540
  presidenti           -2.472
  barack               -2.310
  told 